In [1]:
import sys; sys.path.insert(0, "../src")
from protgen.models.transformer_rocha import Transformer

In [2]:
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
import torch.optim as optim

In [4]:
import matplotlib.pyplot as plt

In [5]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [6]:
val_loader = DataLoader(val_data, batch_size=32, shuffle=True)

In [ ]:
transformer = Transformer(num_heads=4, num_layers=2, vocabulary_size=23, embedding_size=32)

c:\Users\marti\OneDrive\Documents\GitHub\02456-autoregressive-project\notebooks\../src\protgen\models\transformer_rocha.py:41: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, norm=nn.LayerNorm(embedding_dim), num_layers=num_layers)


In [8]:
print(list(transformer.named_parameters()))

[('embedding.weight', Parameter containing:
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [-8.4415e-01, -3.4543e-01,  1.7394e-01, -5.0236e-01, -1.3815e-01,
         -1.6665e+00, -1.1945e+00, -1.0102e+00,  5.0705e-01, -1.1965e+00,
         -1.3162e-01,  2.4783e-01,  1.4396e-01,  3.0309e-01, -1.3606e+00,
          7.1156e-01,  2.7400e-01, -7.6634e-01,  9.0318e-01, -1.3042e+00,
          1.3333e+00,  2.5012e-01, -1.1196e-03,  2.0412e+00,  1.1119e+00,
         -2.5665e-01, -2.4234e-01,  7.7721e-01, -1.2885e+00, -1.1540e+00,
          2.6105e-01, -6.7983e-0

In [9]:
train_data.df

,sequence,family
0,MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHS...,HPr
1,MNKHELITRISEKSGMTKKDVESVLNGFMDEVTDALGSGNKVQLIG...,HU
2,MTQRFEDTFLIQNKLGLHARAASVLAELSAQFDANITLFQGSKEAE...,HPr
3,MNKTELILRIAEETGFSRKNVELGLNAAINLITKALTEEEKVQIVG...,HU
4,MVERKAVIKNEAGIHCRPTAVITQEAAKVDAVIMILSPGGTCQLGS...,HPr
...,...,...
16015,MTKADIINEIANETGVAKKDVTVVVESFMATIKDSLLEKRECVYLR...,HU
16016,MSRLSRQVTIVNQKGLHARASAKFVTFVSRLPEGLKVEVEKDGQCV...,HPr
16017,MNKSDLINHVAAETGLTKADSGFALEAVLEGITKSLRKGDTVTLTG...,HU
16018,MTLTKAELADLLFEKVGLNKREAKDMVEAFFEEIRNALERGEGVKL...,HU


In [10]:
val_data.df

,sequence,family
0,MNKADFVKHVAQQHQCTQIEAEKTIDMFTSSVIDAMGKGNEISLVG...,HU
1,MSERKATIASRVGLHARPASIFAAAVGELPIEVTIGLEADPVEDAM...,HPr
2,MNKSQLIDKIAAGAELSKADAGRALDAFIGAVTDALKEGDSVALVG...,HU
3,MNKSELVDKMAEKSGLSKKDSDKALNAMLESVEEALTSGDKVQLIG...,HU
4,MNKTELIDHIASKADISKAAAGRTLDALIGAVKTTLKKGGTVTLVG...,HU
...,...,...
1996,MDNRSRREAPEAVDTHGYYKAEIEVTHGFGLHARPSVIFTRLAKSF...,HPr
1997,MNKSELIDAIAASADIPKAVAGRALDAVIDSVTGALKAGDSVVLVG...,HU
1998,MSEHESAGRKLVSVDFEIFGYVQGVCFRMYTEREGLRLGLVGWVKN...,AcyP
1999,MNKSELIDAIAESADISKAAAGRALDAMTNTITGALKNDDQVTLIG...,HU


In [11]:
test_data.df

,sequence,family
0,MTKAEVIVAIADKTGVDKASVSTTLESFFEIVKDTLAQGDAVYVRG...,HU
1,MERVTAVARGRVQGVGYRYYVSECAHATGIHGWVKNEPDGTVRIVA...,AcyP
2,MPQSSRTVLIQNRLGLHARAAVKLVELAQSFEAVLTIQNQEGKEAT...,HPr
3,MKHLNLKIHGKVQGVFFREKTKQLAQELGLNGYVKNTSDNCVEVVV...,AcyP
4,MTDRTRAHVYVSGKVQGVYFRATTREEAREQGVDGWVRNLDDGRVE...,AcyP
...,...,...
1996,MNKSELIQAVAERSELTKKDAGSAIQAAFDIISETLEKGETVQLIG...,HU
1997,MANKAELIDSVASKTGLTKKDATSAVDAVFETIQDNLSEGNKVQLI...,HU
1998,MPKLERQVTICNKLGLHARAATKLVVLASEFDASITLIQGEKQASA...,HPr
1999,MAAASAGGPSDPPTPQQSSPTKAVRVVVKGRVQGVFFRDWTVETAR...,AcyP


***CHECK WINDOWING IS WORKING - Transformer should only be able to look at the previous sequence of the token it is predicting***

In [12]:
transformer(torch.tensor([[1, 2, 3, 4, 5]]))

tensor([[[ 0.1204, -0.5898, -0.3277,  0.7371,  0.9988,  0.3569,  0.0296,
          -0.1598, -0.3266,  0.9934,  0.4892, -0.2917, -0.6879,  0.2163,
          -0.0718,  0.4475,  0.1425,  0.4194, -0.3212,  0.0762, -0.9722,
           1.0494,  1.0591],
         [-0.5210, -0.4526, -0.9154,  0.8782,  1.0815, -0.0986, -1.6900,
          -0.0394, -0.5803,  1.5185,  0.5157, -0.0966, -0.3743,  0.4561,
          -0.1866,  0.5178, -0.4269, -0.0080, -0.8603,  0.0414, -0.0963,
           1.2071, -0.1772],
         [-0.2678, -0.5807, -0.8943, -0.0474, -0.4236, -0.2521, -0.3475,
           0.1618, -0.6490,  0.8996,  0.4226, -0.4362, -0.9767,  0.1202,
          -1.3850,  0.6204,  0.3621, -1.1839, -0.1017,  0.9228, -1.3385,
           0.3112, -0.9627],
         [-0.0109,  0.1157, -0.4974, -0.4335,  0.1452,  0.4691,  0.6690,
          -1.3906, -0.1142,  0.0567,  0.4902, -0.9953,  0.2240, -0.1391,
          -1.1029,  0.2718,  0.3441, -0.5130, -0.5489,  0.2546, -0.4999,
           0.1850,  0.3678],
        

In [13]:
loss_fn = nn.CrossEntropyLoss(ignore_index = -1)
optimizer = optim.AdamW(transformer.parameters(), lr= 0.001)

In [14]:
# Test the forward pass with dummy data (from assignments!)
out = transformer(torch.randint(0,23,(32,129)))
print("Output shape:", out.size())
print(f"Output logits:\n{out.detach().cpu().numpy()}")
print(f"Output probabilities:\n{out.softmax(-1).detach().cpu().numpy()}")

Output shape: torch.Size([32, 129, 23])
Output logits:
[[[-2.6909894e-01 -1.1058440e+00 -5.1621789e-01 ... -8.1599909e-01
   -9.5637873e-02  6.7467999e-01]
  [-8.1122899e-01 -1.1417581e+00 -4.6180713e-01 ... -7.2298282e-01
    1.1377382e+00 -2.1652062e-01]
  [-3.1121813e-02 -5.3713685e-01 -3.9726156e-01 ... -1.9098045e-01
    7.1398574e-01 -1.8103099e-01]
  ...
  [-6.0353118e-01 -6.7750776e-01 -7.1244991e-01 ...  6.5552133e-01
    1.9938588e+00 -4.3514061e-01]
  [-4.0412804e-01 -5.0663131e-01 -2.5615087e-01 ...  1.4150815e+00
    1.3818824e+00  2.7126163e-01]
  [ 1.5876144e-02 -7.3910102e-02 -1.1523256e+00 ... -5.4948324e-01
    8.3747834e-01 -4.3936676e-01]]

 [[ 4.0383950e-01 -1.3900051e-01  6.9871938e-01 ... -1.8742366e-01
    7.8199995e-01  8.9905426e-02]
  [ 5.0756544e-02 -9.6086055e-01  4.5240343e-01 ... -5.5425927e-02
    2.9099196e-01  5.5345833e-01]
  [-5.5148280e-01 -3.1360146e-01  2.2980972e-01 ...  1.2322382e-01
    6.6588217e-01 -3.5642332e-01]
  ...
  [-7.2133982e-01 -1.3

In [15]:
print(len(train_loader))

501


In [16]:
num_epochs = 10
val_losses = []
train_losses = []

for epoch in range(num_epochs):
    
    ### Training
    transformer.train()
    total = 0
    for x, x_shifted, y, length in train_loader:
        optimizer.zero_grad()
        out = transformer(x)
        loss = loss_fn(out.transpose(1, 2), x_shifted)
        loss.backward()
        optimizer.step()
        total += loss.item()
    train_loss = total / len(train_loader)
    train_losses.append(train_loss)

    # validation
    transformer.eval()
    total = 0
    with torch.no_grad():
        for x, x_shifted, y, length in val_loader:
            out = transformer(x)
            loss = loss_fn(out.transpose(1, 2), x_shifted)
            total += loss.item()
    val_loss = total / len(val_loader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | train {train_loss:.4f} | val {val_loss:.4f}")

plt.figure()
plt.plot(epoch, train_loss, 'r', epoch, val_loss, 'b')
plt.legend(['Train Loss','Validation Loss'])
plt.xlabel('Updates'), 
plt.ylabel('Loss')

KeyboardInterrupt: 